# Sample adequacy

In [2]:
# ============================================================
# CB-SEM sample-adequacy analysis - all four manuscripts (single cell)
# RMSEA-based power (MacCallum, Browne & Sugawara, 1996) + regression-based check
# Requires: scipy  (pip install scipy)
# ============================================================
from scipy.stats import ncx2, ncf, f as f_dist
from scipy.optimize import brentq

# ---------------- 1. CORE FUNCTIONS ----------------
def sem_df(items, exog, endo, paths):
    """Degrees of freedom and free parameters for a latent-variable model.
    Assumes marker-variable scaling, no cross-loadings, no correlated errors,
    uncorrelated disturbances, and all exogenous latents correlated."""
    for s, t in paths:
        assert s in items and t in items, f"Unknown construct in path {s}->{t}"
        assert t in endo, f"Path target {t} must be endogenous"
    k, p = len(items), sum(items.values())
    moments = p * (p + 1) // 2
    measurement = (p - k) + p                       # free loadings + error variances
    ne = len(exog)
    latent = ne * (ne + 1) // 2 + len(endo) + len(paths)
    npar = measurement + latent
    cfa_npar = measurement + k * (k + 1) // 2
    return dict(indicators=p, npar=npar, df=moments - npar, cfa_df=moments - cfa_npar)

def rmsea_power(df, n, eps0=0.05, epsa=0.08, alpha=0.05):
    """epsa > eps0: test of close fit; epsa < eps0: test of not-close fit."""
    l0, la = (n - 1) * df * eps0**2, (n - 1) * df * epsa**2
    if epsa > eps0:
        return 1 - ncx2.cdf(ncx2.ppf(1 - alpha, df, l0), df, la)
    return ncx2.cdf(ncx2.ppf(alpha, df, l0), df, la)

def min_n(df, target=0.80, **kw):
    g = lambda n: rmsea_power(df, n, **kw) - target
    return 5 if g(5) >= 0 else brentq(g, 5, 200000)

def reg_min_n(u, f2=0.15, alpha=0.05, power=0.90):
    """Regression-based minimum N (fixed-model R-squared test, as in G*Power)."""
    def pw(n):
        d2 = n - u - 1
        crit = f_dist.ppf(1 - alpha, u, d2)
        return 1 - ncf.cdf(crit, u, d2, f2 * n) - power
    return int(brentq(pw, u + 3, 2000)) + 1

def max_predictors(paths, moderated_target):
    """Largest number of predictors in any one equation (+1 for the interaction term)."""
    counts = {}
    for s, t in paths:
        counts[t] = counts.get(t, 0) + 1
    counts[moderated_target] = counts.get(moderated_target, 0) + 1
    return max(counts.values()), counts

# ---------------- 2. MODEL DEFINITIONS ----------------
# Edit item counts marked CHECK. 'moderated' = outcome whose equation has the interaction term.
MODELS = {
    "M2": dict(n=402,
        items={"BRI": 5, "RS": 5, "DI": 5, "SCCE": 5, "RV": 5, "SCR": 5},
        exog=["BRI", "RV"], endo=["RS", "DI", "SCCE", "SCR"],
        paths=[("BRI","RS"), ("BRI","DI"), ("RS","SCCE"), ("DI","SCCE"), ("SCCE","SCR"), ("RV","SCR")],
        direct=[("BRI","SCCE"), ("BRI","SCR"), ("RS","SCR"), ("DI","SCR")],
        moderated="SCR"),
 
}

# ---------------- 3. RUN ----------------
ALPHA = 0.05
for name, m in MODELS.items():
    print(f"\n=== {name}  (N = {m['n']}) ===")
    for label, extra in (("Hypothesised paths", []), ("+ direct paths", m["direct"])):
        paths = m["paths"] + extra
        r = sem_df(m["items"], m["exog"], m["endo"], paths)
        n, u = m["n"], max_predictors(paths, m["moderated"])[0]
        print(f"{label:<19} indicators={r['indicators']:>3}  npar={r['npar']:>4}  df={r['df']:>5}  "
              f"power(close)={rmsea_power(r['df'], n, 0.05, 0.08, ALPHA):.3f}  "
              f"minN(.80)={min_n(r['df'], 0.80, eps0=0.05, epsa=0.08, alpha=ALPHA):.0f}  "
              f"N:npar={n / r['npar']:.2f}  | max predictors={u}  regression minN={reg_min_n(u)}")


=== M2  (N = 402) ===
Hypothesised paths  indicators= 30  npar=  67  df=  398  power(close)=1.000  minN(.80)=56  N:npar=6.00  | max predictors=3  regression minN=99
+ direct paths      indicators= 30  npar=  71  df=  394  power(close)=1.000  minN(.80)=56  N:npar=5.66  | max predictors=6  regression minN=123


In [2]:
2+3

5